# M2 · Baseline reproduction — Rajaraman et al. (2018), PeerJ 6:e4568

**Base paper:** *Pre-trained convolutional neural networks as feature extractors toward improved malaria parasite detection in thin blood smear images.*

**What we reproduce:** a frozen ImageNet **ResNet-50** used as a feature extractor (layer before the classifier, 2,048 values per cell) + a small fully connected classifier with dropout 0.5, evaluated with **5-fold patient-level cross-validation** on the NIH/NLM malaria cell dataset (27,558 cells, 200 patients).

**Target numbers (paper Table 2, ResNet-50, patient level)**

| Accuracy | AUC | Sensitivity | Specificity | F1 | MCC |
|---|---|---|---|---|---|
| 0.957 ± 0.007 | 0.990 ± 0.004 | 0.945 ± 0.020 | 0.969 ± 0.009 | 0.957 ± 0.008 | 0.912 ± 0.014 |

We also run the **same pipeline with a random cell-level split** to measure the cell-vs-patient gap (paper Table 6 reports 0.986 cell-level accuracy). That gap motivates our M3 hypothesis.

**How to run:** Colab → *Runtime → Change runtime type → T4 GPU* → *Run all*. Kaggle also works (turn **Internet on**; the dataset `iarunava/cell-images-for-detecting-malaria` is picked up automatically if attached). Results are written once to `outputs/baseline_v1/` and never overwritten.

## 0 · Configuration and seeds

In [ ]:
import os, sys, json, re, random, time, zipfile, urllib.request, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score, f1_score,
                             matthews_corrcoef, confusion_matrix, roc_curve)

CONFIG = {
    "run_name": "baseline_v1",
    "seed": 42,
    "data_root": "data/cell_images",          # must contain Parasitized/ and Uninfected/
    "data_url": "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip",
    "patient_csv_urls": {
        "Parasitized": "https://data.lhncbc.nlm.nih.gov/public/Malaria/patientid_cellmapping_parasitized.csv",
        "Uninfected":  "https://data.lhncbc.nlm.nih.gov/public/Malaria/patientid_cellmapping_uninfected.csv",
    },
    # Feature extractor (as in the paper)
    "backbone": "ResNet50",
    "backbone_weights": "imagenet",
    "img_size": 224,                          # ResNet-50 input size used in the paper
    "feature_batch": 128,
    # Classifier head (paper: FC model + dropout 0.5, SGD + Nesterov, momentum 0.9, L2 1e-6)
    "head_units": 256,                        # not stated in the paper -> our choice (logged)
    "dropout": 0.5,
    "l2": 1e-6,
    "momentum": 0.9,
    "learning_rate": 1e-3,                    # paper used 1e-6; see deviations table
    "batch_size": 64,
    "max_epochs": 100,
    "early_stop_patience": 8,
    "val_fraction": 0.10,                     # inner validation split, taken from training patients only
    "n_folds": 5,
    "threshold": 0.5,
}

OUT = Path("outputs") / CONFIG["run_name"]
FIG = OUT / "figures"
if (OUT / "summary.json").exists():
    raise SystemExit(f"{OUT} is already frozen. Change CONFIG['run_name'] to start a new run.")
FIG.mkdir(parents=True, exist_ok=True)
CACHE = Path("cache"); CACHE.mkdir(exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

set_seed(CONFIG["seed"])
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Determinism not enabled:", e)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (this will be slow)")

## 1 · Get the data (NIH/NLM single-cell images + official patient-ID mapping)

In [ ]:
DATA_ROOT = Path(CONFIG["data_root"])
KAGGLE_ROOT = Path("/kaggle/input/cell-images-for-detecting-malaria/cell_images")

def has_classes(p):
    return (p / "Parasitized").is_dir() and (p / "Uninfected").is_dir()

if not has_classes(DATA_ROOT):
    if has_classes(KAGGLE_ROOT):
        DATA_ROOT = KAGGLE_ROOT
        print("Using Kaggle copy:", DATA_ROOT)
    else:
        DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        zip_path = DATA_ROOT.parent / "cell_images.zip"
        if not zip_path.exists():
            print("Downloading ~337 MB from NLM ...")
            urllib.request.urlretrieve(CONFIG["data_url"], zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA_ROOT.parent)
print("Data root:", DATA_ROOT)

# Official patient-ID -> cell filename mapping (used to build patient-level folds)
patient_map = {}
for cls, url in CONFIG["patient_csv_urls"].items():
    dest = DATA_ROOT.parent / Path(url).name
    try:
        if not dest.exists():
            urllib.request.urlretrieve(url, dest)
        for line in dest.read_text().splitlines():
            pid = line.split(",")[0].strip()
            for fname in re.findall(r"'([^']+\.png)'", line):
                patient_map[fname] = pid
    except Exception as e:
        print(f"Could not load {cls} mapping ({e}); falling back to the filename token.")
print(f"Official mapping covers {len(patient_map):,} filenames")

## 2 · Index every cell with its label and patient

In [ ]:
rows = []
for cls, label in [("Parasitized", 1), ("Uninfected", 0)]:
    for p in sorted((DATA_ROOT / cls).rglob("*.png")):
        rows.append({"path": str(p), "file": p.name, "class": cls, "label": label})
df = (pd.DataFrame(rows)
        .drop_duplicates(subset=["file", "class"])    # the Kaggle copy nests a duplicate folder
        .sort_values(["class", "file"]).reset_index(drop=True))

# Patient ID: official mapping first; otherwise the slide token at the start of the filename
df["patient_token"] = df["file"].str.split("_").str[0]
df["patient"] = df["file"].map(patient_map).fillna(df["patient_token"])
mapped = df["file"].isin(patient_map.keys()).mean()

print(f"Cells: {len(df):,}  (expected 27,558)")
print(df["class"].value_counts().to_string())
print(f"Patients: {df['patient'].nunique()}  | cells mapped by official CSV: {mapped:.1%}")
print("Cells per patient: median", int(df.groupby('patient').size().median()),
      "| min", df.groupby('patient').size().min(), "| max", df.groupby('patient').size().max())
assert len(df) == 27558, "Unexpected number of cells - check the download."

## 3 · Look at the data

Each row is one patient. Colour is roughly constant within a row and changes between rows: this is the inter-patient staining variation the paper blames for its patient-level drop, and the variable our M3 hypothesis targets.

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
big = df.groupby("patient").filter(lambda g: (g.label == 1).sum() >= 3 and (g.label == 0).sum() >= 3)
show_pat = rng.choice(big["patient"].unique(), size=min(6, big["patient"].nunique()), replace=False)
fig, axes = plt.subplots(len(show_pat), 6, figsize=(9, 1.6 * len(show_pat)))
axes = np.atleast_2d(axes)
for r, pid in enumerate(show_pat):
    g = big[big.patient == pid]
    pick = pd.concat([g[g.label == 1].sample(3, random_state=0), g[g.label == 0].sample(3, random_state=0)])
    for c, (_, row) in enumerate(pick.iterrows()):
        axes[r, c].imshow(plt.imread(row.path)); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title("Parasitized" if row.label else "Uninfected", fontsize=8)
    axes[r, 0].text(-8, 50, pid, fontsize=7, ha="right", va="center")
plt.suptitle("Six patients x (3 parasitized, 3 uninfected) cells", fontsize=10)
plt.tight_layout(); plt.savefig(FIG / "cells_by_patient.png", dpi=150); plt.show()

## 4 · Feature extraction: frozen ImageNet ResNet-50 (layer before the classifier)

In [ ]:
S = CONFIG["img_size"]
prep = tf.keras.applications.resnet50.preprocess_input   # the normalisation these weights were trained with

def load(path):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (S, S), method="bilinear")
    return prep(img)

feat_file = CACHE / f"resnet50_{S}px_{CONFIG['backbone_weights']}.npz"
if feat_file.exists() and list(np.load(feat_file, allow_pickle=True)["files"]) == list(df["file"]):
    X = np.load(feat_file)["X"]; print("Loaded cached features", X.shape)
else:
    backbone = tf.keras.applications.ResNet50(include_top=False, pooling="avg",
                                              weights=CONFIG["backbone_weights"], input_shape=(S, S, 3))
    backbone.trainable = False
    ds = (tf.data.Dataset.from_tensor_slices(df["path"].values)
            .map(load, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(CONFIG["feature_batch"]).prefetch(tf.data.AUTOTUNE))
    t0 = time.time()
    X = backbone.predict(ds, verbose=1).astype("float32")
    np.savez_compressed(feat_file, X=X, files=df["file"].values)
    print(f"Extracted {X.shape} in {time.time() - t0:.0f}s")
y = df["label"].values
groups = df["patient"].values

## 5 · Classifier head and metrics

In [ ]:
def build_head(dim):
    reg = tf.keras.regularizers.l2(CONFIG["l2"])
    m = tf.keras.Sequential([
        tf.keras.Input(shape=(dim,)),
        tf.keras.layers.Dense(CONFIG["head_units"], activation="relu", kernel_regularizer=reg),
        tf.keras.layers.Dropout(CONFIG["dropout"]),
        tf.keras.layers.Dense(2, activation="softmax", kernel_regularizer=reg),
    ])
    m.compile(optimizer=tf.keras.optimizers.SGD(CONFIG["learning_rate"], momentum=CONFIG["momentum"], nesterov=True),
              loss="sparse_categorical_crossentropy")
    return m

def metrics(y_true, p_pos):
    y_hat = (p_pos >= CONFIG["threshold"]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_hat), "auc": roc_auc_score(y_true, p_pos),
            "sensitivity": recall_score(y_true, y_hat), "specificity": tn / (tn + fp),
            "f1": f1_score(y_true, y_hat), "mcc": matthews_corrcoef(y_true, y_hat)}

def run_cv(splitter, split_groups, protocol):
    # Same head, same seeds, only the split changes between protocols
    fold_rows, oof = [], np.zeros(len(y))
    for k, (tr, te) in enumerate(splitter.split(X, y, split_groups)):
        set_seed(CONFIG["seed"] + k)
        if protocol == "patient":   # inner validation = held-out training PATIENTS
            inner = GroupShuffleSplit(1, test_size=CONFIG["val_fraction"], random_state=CONFIG["seed"])
            a, b = next(inner.split(tr, y[tr], groups[tr]))
        else:                       # inner validation = random training CELLS
            a, b = train_test_split(np.arange(len(tr)), test_size=CONFIG["val_fraction"],
                                    stratify=y[tr], random_state=CONFIG["seed"])
        fit_idx, val_idx = tr[a], tr[b]
        model = build_head(X.shape[1])
        model.fit(X[fit_idx], y[fit_idx], validation_data=(X[val_idx], y[val_idx]),
                  epochs=CONFIG["max_epochs"], batch_size=CONFIG["batch_size"], verbose=0,
                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=CONFIG["early_stop_patience"],
                                                              restore_best_weights=True)])
        p = model(X[te], training=False).numpy()[:, 1]
        oof[te] = p
        m = metrics(y[te], p)
        m.update({"fold": k + 1, "test_patients": len(set(groups[te])),
                  "test_parasitized": int(y[te].sum()), "test_uninfected": int((1 - y[te]).sum()),
                  "patient_overlap": len(set(groups[te]) & set(groups[tr]))})
        fold_rows.append(m)
        print(f"[{protocol}] fold {k+1}: acc {m['accuracy']:.3f}  auc {m['auc']:.3f}  "
              f"sens {m['sensitivity']:.3f}  spec {m['specificity']:.3f}  overlap {m['patient_overlap']}")
    return pd.DataFrame(fold_rows), oof

METRICS = ["accuracy", "auc", "sensitivity", "specificity", "f1", "mcc"]

## 6 · Protocol A — 5-fold **patient-level** CV (the reproduction)

No patient appears in both the training and test side of a fold (`overlap` must be 0). The paper's exact fold assignment is not published, so folds are rebuilt from patient IDs with a fixed seed; fold sizes are printed for comparison with paper Table 1 (≈2,700 cells per class per fold).

In [ ]:
patient_cv = StratifiedGroupKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
folds_patient, oof_patient = run_cv(patient_cv, groups, "patient")
assert folds_patient["patient_overlap"].sum() == 0, "Patient leakage between train and test!"
folds_patient[["fold", "test_patients", "test_parasitized", "test_uninfected"]]

## 7 · Protocol B — 5-fold **cell-level** CV (same pipeline, random split)

In [ ]:
cell_cv = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
folds_cell, oof_cell = run_cv(cell_cv, None, "cell")
print("Mean patients shared between train and test per fold:", folds_cell["patient_overlap"].mean())

## 8 · Results against the paper

In [ ]:
paper_patient = {"accuracy": (0.957, 0.007), "auc": (0.990, 0.004), "sensitivity": (0.945, 0.020),
                 "specificity": (0.969, 0.009), "f1": (0.957, 0.008), "mcc": (0.912, 0.014)}
paper_cell = {"accuracy": 0.986, "auc": 0.999, "sensitivity": 0.981, "specificity": 0.992, "f1": 0.987, "mcc": 0.972}

def ms(s): return f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
table = pd.DataFrame({
    "Paper · patient (Table 2)": {m: f"{v[0]:.3f} ± {v[1]:.3f}" for m, v in paper_patient.items()},
    "Ours · patient": {m: ms(folds_patient[m]) for m in METRICS},
    "Δ (ours − paper)": {m: f"{folds_patient[m].mean() - paper_patient[m][0]:+.3f}" for m in METRICS},
    "Ours · cell": {m: ms(folds_cell[m]) for m in METRICS},
    "Paper · cell (Table 6)": {m: f"{paper_cell[m]:.3f}" for m in METRICS},
    "Gap: ours cell − ours patient": {m: f"{folds_cell[m].mean() - folds_patient[m].mean():+.3f}" for m in METRICS},
})
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
cm = confusion_matrix(y, (oof_patient >= CONFIG["threshold"]).astype(int), labels=[0, 1])
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
ax[0].set_xticks([0, 1], ["Uninfected", "Parasitized"]); ax[0].set_yticks([0, 1], ["Uninfected", "Parasitized"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True"); ax[0].set_title("Patient-level CV, pooled out-of-fold")

for name, oof, ls in [("Patient-level", oof_patient, "-"), ("Cell-level", oof_cell, "--")]:
    fpr, tpr, _ = roc_curve(y, oof)
    ax[1].plot(fpr, tpr, ls, label=f"{name} (AUC {roc_auc_score(y, oof):.3f})")
ax[1].plot([0, 1], [0, 1], ":", color="grey"); ax[1].set_xlim(0, 0.3); ax[1].set_ylim(0.7, 1.0)
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("Sensitivity"); ax[1].legend(); ax[1].set_title("ROC (zoomed)")
plt.tight_layout(); plt.savefig(FIG / "confusion_and_roc.png", dpi=150); plt.show()

## 9 · Deviations from the paper (kept deliberately small)

| Item | Paper | Ours | Why |
|---|---|---|---|
| Framework | Keras 2.1.1 / TensorFlow 1.4 | TensorFlow 2.x Keras | 2018 stack no longer installs on Colab |
| Head learning rate | 1e-6 (after random search) | 1e-3, early stopping on held-out training patients | 1e-6 needs thousands of epochs; same SGD + Nesterov, momentum 0.9, L2 1e-6 |
| Head width | not stated | Dense 256 + dropout 0.5 | smallest standard choice |
| Fold assignment | not published | StratifiedGroupKFold on official patient IDs, seed 42 | rebuilt; sizes printed above |
| Hyper-parameter search | randomized grid search | none (paper's reported optimum used) | time budget; avoids tuning on test folds |
| Resize interpolation | not stated | bilinear, 224 × 224 | Keras default |

## 10 · Freeze the results

In [ ]:
folds_patient.to_csv(OUT / "metrics_per_fold_patient.csv", index=False)
folds_cell.to_csv(OUT / "metrics_per_fold_cell.csv", index=False)
table.to_csv(OUT / "comparison_table.csv")
pd.DataFrame({"file": df["file"], "patient": df["patient"], "label": y,
              "p_parasitized_patientCV": oof_patient, "p_parasitized_cellCV": oof_cell}
            ).to_csv(OUT / "oof_predictions.csv", index=False)
fold_of = {}
for k, (_, te) in enumerate(patient_cv.split(X, y, groups)):
    fold_of.update({g: k + 1 for g in set(groups[te])})
pd.Series(fold_of, name="fold").rename_axis("patient").to_csv(OUT / "patient_folds.csv")

summary = {
    "base_paper": "Rajaraman et al., PeerJ 6:e4568 (2018), ResNet-50 feature extractor, Table 2",
    "n_cells": int(len(df)), "n_patients": int(df["patient"].nunique()),
    "patient_level": {m: {"mean": float(folds_patient[m].mean()), "sd": float(folds_patient[m].std(ddof=1))} for m in METRICS},
    "cell_level": {m: {"mean": float(folds_cell[m].mean()), "sd": float(folds_cell[m].std(ddof=1))} for m in METRICS},
    "paper_patient_level": {m: v[0] for m, v in paper_patient.items()},
    "created": time.strftime("%Y-%m-%d %H:%M"),
}
env = {"python": platform.python_version(), "tensorflow": tf.__version__, "numpy": np.__version__,
       "pandas": pd.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
       "platform": platform.platform()}
(OUT / "config.json").write_text(json.dumps(CONFIG, indent=2))
(OUT / "env.json").write_text(json.dumps(env, indent=2))
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))   # written last = run is frozen

pl, cl = summary["patient_level"], summary["cell_level"]
print("FOR THE SLIDES")
print(f"Patient-level (ours): acc {pl['accuracy']['mean']:.3f}±{pl['accuracy']['sd']:.3f} | "
      f"AUC {pl['auc']['mean']:.3f} | sens {pl['sensitivity']['mean']:.3f} | spec {pl['specificity']['mean']:.3f} | "
      f"F1 {pl['f1']['mean']:.3f} | MCC {pl['mcc']['mean']:.3f}")
print(f"Cell-level (ours):    acc {cl['accuracy']['mean']:.3f} | sens {cl['sensitivity']['mean']:.3f}")
print(f"Cell − patient accuracy gap: {cl['accuracy']['mean'] - pl['accuracy']['mean']:+.3f}")
print("Frozen to", OUT.resolve())